In [1]:
##Importante tener cuenta en hugging face, para algunos modelos como meta/llama
# se debe tener aprobacion para descargar desde la plataforma el modelo. se debe ingresar a la pagina
#https://huggingface.co/google/llama3.2/1b y aceptar los terminos del modelo para poderlo descargar con la cuenta de hugginface
from huggingface_hub import login
login(new_session=False)

In [3]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

ADAPTER_DIR = "models/llama3.2-1b"   #carpeta local del LoRA/adaptador
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
HF_TOKEN = os.getenv("HF_TOKEN")

# Descubre el base exacto que espera el adaptador
base_id = PeftConfig.from_pretrained(ADAPTER_DIR).base_model_name_or_path
print("Base esperado por el LoRA:", base_id)

BASE_ID = base_id or "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(BASE_ID, use_fast=True, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE, device_map="auto",
                                             token=HF_TOKEN, trust_remote_code=True)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()


Base esperado por el LoRA: meta-llama/Llama-3.2-1B


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

C:\Users\jsoa\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jsoa\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(


In [5]:
# ============================
# Generar PLS con CoT Implicito Qwen (HF) usando columnas: name, article, summary
# ============================

import re
import time  # NEW
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # usa EOS como PAD
tokenizer.padding_side = "left"    
@torch.inference_mode()
def generate_pls_batch(
    data: str,
    prompt_fn,
    batch_size: int = 2,
    max_new_tokens: int = 380,     # suficiente para 4–6 oraciones
    temperature: float = 0.0,      # determinista → mejor factualidad
    top_p: float = 1.0,
    num_beams: int = 1,
    repetition_penalty: float = 1.02,
    no_repeat_ngram_size: int = 4,
):
    texts = data['article'].fillna("").astype(str).tolist()
    df_out = data.copy()

    # Calcula un input máximo seguro
    max_ctx = getattr(model.config, "max_position_embeddings", 4096)
    max_input_len = max(8, max_ctx - max_new_tokens)

    outputs = []
    latencies = []  # NEW
    pbar = tqdm(total=len(texts), desc='Generando resúmenes', unit="sample")

    try:
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            prompts = [prompt_fn(t) for t in batch_texts]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_input_len,
            )
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device)

            # --- NEW: medir tiempo del batch con sync de GPU
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0.0 or top_p < 1.0),
                temperature=temperature,
                top_p=top_p,
                num_beams=num_beams,
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            per_item_latency = (t1 - t0)  # NEW

            # Cortar por muestra usando la longitud REAL (no el ancho padded)
            batch_out = []
            input_lens = attention_mask.sum(dim=1)  # [B]
            for row in range(input_ids.size(0)):
                ilen = int(input_lens[row].item())
                gen_only = gen_ids[row, ilen:]        # ← clave: corta desde fin del prompt de ESA fila
                text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
                batch_out.append(text)

            outputs.extend(batch_out)
            latencies.extend([per_item_latency] * len(batch_texts))  # NEW
            pbar.update(len(batch_texts))
    finally:
        pbar.close()

    df_out['gen_summary'] = outputs
    df_out['latency_s'] = latencies  # NEW
    return df_out

# O el CoT factual corto
def prompt_cot_factual(t):
    return f"""You are a helpful medical writer.
            Think briefly before answering:
            - Use only statements explicitly present in the source.
            - Keep names and numbers exactly as written.
            - 4–6 sentences, ≤120 words. Do not show your reasoning.

            Scientific text:
            {t}

            Plain summary:"""

DATA_DIR = Path("data-sources/pre-processed")
RESULTS_DIR = Path("models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_test = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
df_output = generate_pls_batch(
    data=df_test,
    prompt_fn=prompt_cot_factual, 
    batch_size=5,
    max_new_tokens=380,
    temperature=0.0,
    top_p=1.0,
)

csv_out = RESULTS_DIR / "summaries_llama32-1b_COT.csv"
df_output.to_csv(csv_out, index=False, encoding="utf-8")
print(f"Guardado CSV con PLS: {csv_out}")
print("Latencia promedio (s):", df_output["latency_s"].mean())

#3.1 a 4.6 GB VRAM



Generando resúmenes:   0%|          | 0/380 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Guardado CSV con PLS: models\results\summaries_llama32-1b_COT.csv
Latencia promedio (s): 30.35941425657912
